# 04 · Tabla de frames y estadísticas por equipo (análisis exploratorio)

**Qué hace este cuaderno.** Construye, a partir del tracking automático (ATD), de los metadatos de equipo y del
Smart Tagging, tres tablas de análisis y las describe con agregados:

1. **`ev`** — eventos (Smart Tagging): una fila por fase/jugada a balón parado.
2. **`slots`** — jugadores/slots: una fila por slot de tracking (46), con su equipo y su ocupación.
3. **`ft`** — frames: una fila por frame de vídeo (199.319) con estadísticas **colectivas** por equipo.

Después estudia distribuciones por equipo, medidas relativas al balón, condiciones de contorno (campo, mitades,
saque inicial) y hace un **clustering exploratorio sin etiquetas**.

**Reglas que este cuaderno respeta (no negociables).**
- Solo se muestran **agregados** (conteos, medias, desviaciones, cuantiles). Nunca filas ni posiciones. La función
  `show()` de abajo rechaza imprimir tablas de frames.
- Las tablas de frames se guardan **fuera del repositorio** (`$MTAH_DATA_DIR/derived/`). Nunca se commitean.
- **Antes de commitear este cuaderno hay que borrar sus salidas**: `jupyter nbconvert --clear-output --inplace notebooks/04_frame_features.ipynb`.
  El hook de commit bloquea cuadernos con salidas.
- El GT público no se usa. Los clusters **no son etiquetas**: pasar de cluster a H/A/D se aprende en partidos
  externos etiquetados, no mirando el partido (RULES.md: no anotación manual).

## 0 · Diccionario de los datos de entrada (para entenderlos sin tener los ficheros)

**Partido.** FC Barcelona (local, `H`) – Athletic Club (visitante, `A`), LaLiga 2024-25, jornada 2. Vídeo *táctico*
de 199.319 frames a 25 fps (1920×1080). Una fila de la entrega = un frame de este vídeo (índice 0..199318).

**Frames puntuados** (el resto se ignora): 1ª parte 11.185–90.651 y 2ª parte 110.808–190.197 (158.857 frames).
Los frames 110.808–113.741 están puntuados pero el balón aún no está en juego (el saque inicial de la 2ª parte es
el frame 113.742; esos frames son `D`). Fuera de esas ventanas: antes del partido, descanso y después del pitido final.

### ATD — tracking automático (`BAR-ATH_tactical_atd.txt` + `_atd.xml`)
- Salida del pipeline de visión por computador de Metrica. **Automático, sin corregir**, una línea por frame.
- Formato de línea: `frame_id:slot;slot;...:balón`, cada grupo `x,y,velocidad`. `frame_id` empieza en 1
  (`frame de vídeo = frame_id − 1`). **Coordenadas normalizadas 0–1**; se pasan a metros multiplicando por el campo
  **105,3 × 68,0 m** (x = largo, y = ancho). La velocidad del fichero viene en m/s (aquí no se usa: se recalcula).
- **46 slots** (`Track 1..Track 46`) para 22 jugadores: aparecen, desaparecen e **intercambian identidad**. No hay
  dorsales ni identidad de jugador, solo posición y **equipo del slot** (metadatos). Por eso solo valen estadísticas
  **colectivas** (centroide, dispersión, velocidades agregadas, distancias al balón).
- Datos que faltan = `NaN`. **El balón falta en ~41 % de los frames puntuados** (visible en 56,9 % de todos los
  frames) y falta sobre todo cuando más importa (cajas llenas, balones aéreos, entradas). Hay ~20,5 objetos
  seguidos por frame de media (a veces > 22 por duplicados) y a veces objetos fuera del campo (árbitros, banquillo).
- **Metadatos (`_atd.xml`)**: lista los dos equipos y asigna cada slot a uno. El primero es el local (FIFA/EPTS).
  Este es el cuadro de **jugadores por equipo**: 24 slots locales y 22 visitantes.

### Smart Tagging (`BAR-ATH_tactical_smart_tagging.xml`)
- Anotación **automática** de fases de juego y jugadas a balón parado (SportsCode XML), 638 instancias. **No es
  verdad terreno**: tiene falsos positivos y negativos. Cada instancia es un intervalo `[start, end]` en segundos
  desde el primer frame del vídeo (`frame = round(segundos × 25)`) con un `code` y etiquetas.
- Códigos: `PLAYERS IN THE BOX`, `BALL IN FINAL THIRD`, `BALL IN THE BOX`, `PROGRESSION`, `DEFENDING IN MIDDLE THIRD`,
  `SET PIECES`, `CREATING CHANCES`, `DEFENDING IN DEFENSIVE THIRD`, `LONG BALL`, `ATTACKING TRANSITION`,
  `DEFENSIVE TRANSITION`, `BUILD UP`, `DEFENDING IN ATTACKING THIRD`, `GOALS`.
- Etiquetas: `Team` (Barcelona / Atletic Club / **N/A**, p. ej. todos los saques de banda), `Half`, `Type`
  (saque de banda, córner, saque de puerta, falta, saque inicial…), `Side`, `Direction of ball entry`, `Max Players in the box`.
- **Los códigos vienen emparejados sobre el mismo intervalo** (p. ej. `PROGRESSION` con `DEFENDING IN MIDDLE THIRD`,
  `BUILD UP` con `DEFENDING IN ATTACKING THIRD`, `CREATING CHANCES` con `DEFENDING IN DEFENSIVE THIRD`,
  `ATTACKING TRANSITION` con `DEFENSIVE TRANSITION`). En los `DEFENDING…` y `DEFENSIVE TRANSITION` el `Team` es el
  equipo que **defiende**; el que posee es el otro (columna `poss_side` de `ev`). Las ventanas de `SET PIECES`
  son casi siempre de **35 s** fijos, así que no marcan con precisión el parón.
- Son **fases**, no posesiones: una fase puede cubrir una posesión, quedarse corta o abarcar dos.

### Homografía (no se usa aquí)
Matriz 3×3 por frame que proyecta el campo normalizado a píxeles; sirve para dibujar sobre el vídeo.

### Etiquetas finales del reto (no visibles aquí)
`H` posesión local, `A` visitante, `D` balón muerto, `X` sin respuesta. Una **posesión** es una racha máxima de `H`
o de `A` con el balón vivo; un `D` la corta. El GT sale de tracking corregido y es privado salvo 5 minutos públicos.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:.3f}".format)

DATA_DIR = os.environ.get("MTAH_DATA_DIR")
assert DATA_DIR, "Define MTAH_DATA_DIR (carpeta con tactical/, fuera del repo)"
DERIVED = Path(DATA_DIR) / "derived"
DERIVED.mkdir(parents=True, exist_ok=True)


def show(df, max_rows=90):
    # Guard: refuse to display anything that looks like frame-level rows.
    assert len(df) <= max_rows and "frame" not in df.columns, "refusing to display frame-level rows"
    display(df)

## 1 · Carga

In [ ]:
from src.competition import load_competition          # also puts starter/ on sys.path
import games
from load_smart_tagging import load as load_tags
from src.model.frame_features import (FULL_DOC, aggregate_table, add_normalised, attack_direction,
                                      build_frame_table)

tr, runs, forced_dead = load_competition(DATA_DIR)
events_raw = load_tags(games.GAME, DATA_DIR)
N = tr.n_frames
graded = np.zeros(N, dtype=bool)
for lo, hi in runs:
    graded[lo:hi] = True

print("frames:", N, "| graded:", int(graded.sum()), "| slots:", tr.players.shape[1],
      "| smart tagging instances:", len(events_raw))
print("graded runs [start, stop):", runs, "| forced-D block (graded, ball not yet in play) frames:",
      [b - a for a, b in forced_dead])

## 2 · Tabla `ev`: eventos del Smart Tagging

Una fila por instancia (**metadatos de entrada automáticos**, no etiquetas del reto). Columnas:
`code` (tipo de fase), `start_frame`/`end_frame` (frames de vídeo, fin exclusivo), `dur_s` (segundos),
`team_tag` (club etiquetado), `half_tag`, `type` (solo en `SET PIECES`), `side`, y **`poss_side`**: el lado
(`H`/`A`) que **posee** según el tag, invirtiendo el equipo en los códigos `DEFENDING…` y `DEFENSIVE TRANSITION`.
`poss_side` es `None` cuando el equipo es `N/A`.

In [ ]:
rows = []
for e in events_raw:
    tags = e["tags"]
    team = (tags.get("Team") or "").strip().lower()
    side = None if team in ("", "n/a") else ("H" if team.startswith("barcel") else "A")
    name = (e["code"] or "").strip().upper()
    if side is not None and (name.startswith("DEFENDING") or name == "DEFENSIVE TRANSITION"):
        side = "A" if side == "H" else "H"
    rows.append({"code": name, "start_frame": e["start_frame"], "end_frame": e["end_frame"],
                 "dur_s": (e["end_frame"] - e["start_frame"]) / 25.0, "team_tag": tags.get("Team"),
                 "half_tag": tags.get("Half"), "type": tags.get("Type"), "side": tags.get("Side"), "poss_side": side})
ev = pd.DataFrame(rows)
print("ev columns/dtypes:"); print(ev.dtypes.to_string())
show(ev.groupby("code").agg(n=("dur_s", "size"), dur_med_s=("dur_s", "median"), dur_max_s=("dur_s", "max"),
                            poss_H=("poss_side", lambda s: int((s == "H").sum())),
                            poss_A=("poss_side", lambda s: int((s == "A").sum())),
                            no_team=("poss_side", lambda s: int(s.isna().sum()))))

## 3 · Tabla `slots`: jugadores (slots) y equipo

Una fila por slot de tracking (46): su `team` (de los metadatos), su **ocupación** (fracción de frames puntuados en
que está visible), cuántas veces aparece (`n_runs`) y la duración mediana de cada aparición (`run_med_s`).
Una ocupación baja y muchas apariciones cortas indican identidad poco fiable: por eso solo usamos estadísticas colectivas.

In [ ]:
from src.profile_datasets import _run_bounds

seen = np.isfinite(tr.players[:, :, 0])
recs = []
for s in range(tr.players.shape[1]):
    starts, lens = [], []
    for lo, hi in runs:
        st, ln = _run_bounds(seen[lo:hi, s])
        lens.append(ln)
    ln = np.concatenate(lens) if lens else np.zeros(0)
    recs.append({"slot": s, "team": "home" if tr.slot_team[s] == 0 else "away",
                 "occupancy_graded": float(seen[graded, s].mean()), "n_runs": int(len(ln)),
                 "run_med_s": float(np.median(ln) / 25.0) if len(ln) else np.nan})
slots = pd.DataFrame(recs)
print("slots columns/dtypes:"); print(slots.dtypes.to_string())
show(slots.groupby("team").agg(n_slots=("slot", "size"), occ_mean=("occupancy_graded", "mean"),
                               occ_p10=("occupancy_graded", lambda s: s.quantile(0.1)),
                               occ_p90=("occupancy_graded", lambda s: s.quantile(0.9)),
                               runs_med=("n_runs", "median"), run_med_s=("run_med_s", "median")))

## 4 · Tabla `ft`: estadísticas por frame

Una fila por frame de vídeo (199.319 filas) con estadísticas **colectivas** calculadas solo con el tracking, sin
etiquetas. Ventana de velocidad: ±12 frames (≈ 1 s) para no amplificar el ruido de posición. Sufijo `_h` = local,
`_a` = visitante; sufijo `_n` = **orientación normalizada** (el local siempre ataca hacia +x; se detecta por mitad de forma
no supervisada). **Esta tabla es un export derivado a nivel de frame: no se commitea ni se imprime.**
Se guarda en `$MTAH_DATA_DIR/derived/frame_features.parquet`.

**Diccionario de columnas** (generado desde el mismo código que las calcula):

In [ ]:
ft = add_normalised(build_frame_table(tr, runs))
print("ft shape:", ft.shape)
assert set(ft.columns) == set(FULL_DOC), "undocumented or missing columns"
show(pd.DataFrame({"description": pd.Series(FULL_DOC)}))

In [ ]:
ft.to_parquet(DERIVED / "frame_features.parquet")           # needs pyarrow; stays outside the repo
print("saved ->", DERIVED / "frame_features.parquet")
G = ft["half"].to_numpy() > 0                                 # graded frames

**Nulos y agregados de todas las columnas** (solo frames puntuados): `null_share` es la fracción de frames sin dato.

In [ ]:
show(aggregate_table(ft, [c for c in ft.columns if c not in ("frame", "half")], mask=G))

## 5 · Distribuciones por equipo (local vs visitante)

Medias y desviaciones por frame de cada estadística de equipo, y su distribución a lo largo del partido.
`spd_*` = velocidad de los jugadores del equipo (m/s), `sx`/`sy` = profundidad/anchura del bloque (m),
`pair` = distancia media entre jugadores del equipo (compacidad, m), `cx`/`cy` = centroide (m).

In [ ]:
base = ["cx", "cy", "sx", "sy", "pair", "spd_mean", "spd_std", "spd_med", "cvx", "cvy", "n"]
show(aggregate_table(ft, [f"{b}_h" for b in base] + [f"{b}_a" for b in base], mask=G))

**Comparación directa local − visitante** (media y desviación de la diferencia frame a frame; positivo = el local
tiene más). Sirve para ver asimetrías sistemáticas (dominio territorial, ritmo, compacidad).

In [ ]:
diff = pd.DataFrame({f"{b}_h_minus_a": ft[f"{b}_h"] - ft[f"{b}_a"] for b in ["cx", "sx", "sy", "pair", "spd_mean", "spd_std"]})
show(aggregate_table(diff, list(diff.columns), mask=G))

## 6 · Medidas relativas al balón

Solo en frames con balón visible (~59 % de los puntuados). Para cada equipo: distancia del balón a su jugador más
cercano (`dball_min`), media de los 3 más cercanos (`dball_3`), distancia al centroide (`dball_cen`) y nº de jugadores
a < 5 m y < 10 m (`near5`, `near10`).

In [ ]:
BV = G & (ft["ball_visible"].to_numpy() == 1)
rel = [f"{b}_{t}" for t in ("h", "a") for b in ("dball_min", "dball_3", "dball_cen", "near5", "near10")]
show(aggregate_table(ft, rel + ["ball_speed"], mask=BV))
closer_h = (ft["dball_min_h"] < ft["dball_min_a"])[BV]
print("share of ball-visible graded frames where the nearest player is home: %.3f" % closer_h.mean())

## 7 · Condiciones de contorno

Las condiciones de contorno son las **reglas del juego y del campo** que acotan el problema (origen: Reglas de Juego
FIFA y tamaño de campo publicado), no valores ajustados a este partido:
- El balón fuera del rectángulo 105,3 × 68 m ⇒ balón fuera de juego (Ley 9): `ball_out`.
- Las mitades puntuadas y el bloque `D` previo al saque inicial de la 2ª parte (publicados).
- El equipo local cambia de lado en la 2ª parte: orientación normalizada (`*_n`).

In [ ]:
print("ball outside the pitch: %.2f%% of ball-visible graded frames" % (100 * ft["ball_out"][BV].mean()))
print("attack direction of the HOME team per half (+1 = towards +x in raw coordinates):", attack_direction(ft))
for h in (1, 2):
    m = ft["half"].to_numpy() == h
    print("half", h, "| frames", int(m.sum()), "| home centroid x (raw) mean %.1f, normalised mean %.1f" % (
        ft["cx_h"][m].mean(), ft["cx_h_n"][m].mean()))

## 8 · Clustering exploratorio de frames (sin etiquetas)

Cada frame es una observación; el espacio de rasgos son los **descriptores colectivos normalizados por orientación**
(no las posiciones crudas, que dependen del orden arbitrario de los slots). Pasos:
1. Solo frames puntuados y completos en los rasgos elegidos. Se ajusta con 1 de cada 10 frames y se asigna el resto.
2. PCA (varianza explicada) y k-means con k = 2…8 y 3 semillas.
3. **Regla fija de elección de k** (decidida antes de mirar): el mayor k cuya concordancia media entre semillas (índice
   de Rand ajustado) sea ≥ 0,7; si ninguno, k = 2. No se elige k "a ojo".
4. Se describe cada cluster **solo con agregados** (tamaño, balón visible, balón fuera, quietud, velocidades, solape con
   ventanas de `SET PIECES`). **No se asigna H/A/D a ningún cluster.**

In [ ]:
from src.model.cluster import adjusted_rand, assign, kmeans, pca, standardise

TEAM_FEATS = ["cx_h_n", "cx_a_n", "cy_h", "cy_a", "sx_h", "sx_a", "sy_h", "sy_a", "pair_h", "pair_a",
              "spd_mean_h", "spd_mean_a", "spd_std_h", "spd_std_a", "cvx_h_n", "cvx_a_n"]
ok = G & ft[TEAM_FEATS].notna().all(axis=1).to_numpy()
X = ft.loc[ok, TEAM_FEATS].to_numpy(dtype=float)
sub = X[::10]
Z, mu, sd = standardise(sub)
_, ratio = pca(Z, 8)
print("frames used: %d of %d graded (%.1f%%) | fit sample: %d" % (ok.sum(), G.sum(), 100 * ok.sum() / G.sum(), len(sub)))
print("PCA explained variance ratio (first 8):", np.round(ratio, 3), "| cumulative:", np.round(ratio.cumsum(), 3))

table, models = [], {}
for k in range(2, 9):
    fits = [kmeans(Z, k, seed=s) for s in range(3)]
    ari = np.mean([adjusted_rand(fits[i][0], fits[j][0]) for i in range(3) for j in range(i + 1, 3)])
    table.append({"k": k, "inertia_seed0": fits[0][2], "mean_ARI_between_seeds": ari})
    models[k] = fits[0]
stab = pd.DataFrame(table).set_index("k")
show(stab)
K = max([k for k in stab.index if stab.loc[k, "mean_ARI_between_seeds"] >= 0.7] or [2])
print("selected k by the fixed rule:", K)

In [ ]:
centers = models[K][1]
labels_all = assign((X - mu) / sd, centers)
sp =np.zeros(N, dtype=bool)                                 # inside a SET PIECES window (Smart Tagging)
for e in ev[ev["code"] == "SET PIECES"].itertuples():
    sp[e.start_frame:e.end_frame] = True

desc = ft.loc[ok, ["ball_visible", "ball_out", "crowd_speed", "spd_mean_h", "spd_mean_a", "pair_h", "pair_a"]].copy()
desc["set_piece_window"] = sp[ok].astype(int)
desc["cluster"] = labels_all
summary = desc.groupby("cluster").agg(share=("ball_visible", lambda s: len(s) / len(desc)),
                                       ball_visible=("ball_visible", "mean"), ball_out=("ball_out", "mean"),
                                       crowd_speed=("crowd_speed", "mean"), spd_h=("spd_mean_h", "mean"),
                                       spd_a=("spd_mean_a", "mean"), pair_h=("pair_h", "mean"),
                                       pair_a=("pair_a", "mean"), in_set_piece_window=("set_piece_window", "mean"))
show(summary)

## 9 · Qué hacer con esto (y qué no)

**Sí:**
- Usar estas estadísticas como **evidencia colectiva** para los frames sin balón (idea 4′): posesión y parón se
  reflejan en la forma y el movimiento de los equipos aunque el balón no se vea.
- Comprobar la **transferencia**: calcular la misma tabla en los partidos externos (degradados) y comparar
  distribuciones con las de aquí. Si se parecen, un clasificador entrenado allí con etiquetas puede usarse aquí.
- Entrenar un clasificador de `H/A/D` **solo en partidos externos etiquetados** con estos rasgos y medir su spellF1 en
  el banco externo con la regla de decisión fijada.

**No:**
- Entrenar ni ajustar nada con el GT público de este partido.
- Poner nombre a un cluster y convertirlo en regla mirando frames del partido.
- Commitear `ft`, el parquet o este cuaderno con salidas.